## ChatPromptTemplate (최신 권장 방식)

> 원본: CH02 `05-ChatPromptTemplate.ipynb`  
> 기준: LangChain 1.x / langchain-core 1.2+ (2026년 9월)

In [ ]:
# 필요 패키지 설치 (최초 1회)
# %pip install -qU langchain langchain-openai langsmith python-dotenv

### 🔄 변경 사항: 환경 설정
- `langchain_teddynote.logging.langsmith()` → LangSmith 공식 환경변수 `LANGSMITH_TRACING`, `LANGSMITH_PROJECT` 직접 설정  
  (써드파티 래퍼 없이 동일하게 동작합니다. `.env` 파일에 넣어 두어도 됩니다.)

In [1]:
import os
from dotenv import load_dotenv

# .env 에 OPENAI_API_KEY, LANGSMITH_API_KEY 를 넣어 두세요.
load_dotenv()

# LangSmith 추적 설정
# (langchain_teddynote.logging.langsmith() 대신, LangSmith 공식 환경변수를 직접 설정)
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "CH02-Prompt"

### 🔄 변경 사항
- 원본에서 import 만 하고 쓰지 않던 `load_prompt` 를 제거했습니다. (게다가 langchain-core 1.2.21 부터 deprecated 입니다.)
- `ChatPromptTemplate.from_messages([...])` 와 `ChatPromptTemplate([...])` 는 동일합니다. 최신 문서는 생성자 형태를 주로 사용합니다.
- 역할은 `("user", ...)` 튜플 외에 OpenAI 스타일 dict 로도 쓸 수 있습니다.

In [2]:
from langchain_core.prompts import ChatPromptTemplate

# ChatPromptTemplate 생성
prompt = ChatPromptTemplate(
    [
        ("system", "You are a helpful assistant"),
        ("user", "{country}의 수도는 어디인가요?"),
    ]
)
prompt

ChatPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?'), additional_kwargs={})])

In [3]:
# 같은 프롬프트를 dict 형식 메시지로 정의할 수도 있습니다.
prompt_dict_style = ChatPromptTemplate(
    [
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "{country}의 수도는 어디인가요?"},
    ]
)
prompt_dict_style.invoke({"country": "대한민국"}).to_messages()

[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='대한민국의 수도는 어디인가요?', additional_kwargs={}, response_metadata={})]

### 모델과 연결해서 실행

In [4]:
from langchain.chat_models import init_chat_model

# 모델은 "provider:model" 문자열 하나로 지정합니다.
# 다른 모델로 바꾸려면 이 한 줄만 수정하면 됩니다. (예: "anthropic:claude-sonnet-4-6")
MODEL = "openai:gpt-5.6-luna"

llm = init_chat_model(MODEL)

In [5]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()
chain.invoke({"country": "대한민국"})

'대한민국의 수도는 서울특별시입니다.'